![Week 3 Day 5](../assets/banners/w3d5.png)

# Week 3 Exercise: Let Emma Edit a Question During the Review

On day 5 you watched GradeWise pause at `admin_review` and wait for Emma's yes or no. But real reviews aren't
yes-or-no. Emma reads the extracted exam and spots a typo in Q4a, or decides that Q3's definition should be worth
1.5 marks rather than 2. Today she shouldn't have to *reject* the exam and re-run the extraction (and pay for
it again!) over a small fix: she should be able to **edit the question right there, during the interrupt**.

You'll build a small review workflow around the real CS201 exam, **fully offline** (no LLM needed: the exam is
already extracted):

![Diagram](../assets/diagrams/2397da0f2f44.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([start]) --> V[validate] --> A[[admin_review ⏸]]
  A -- "edited: check again" --> V
  A -- "bad edit: ask again" --> A
  A -- approved --> D[(store)]
  A -- rejected --> X([end])
```

</details>


Emma resumes with one of:
- `{"approved": True}` or `{"approved": False}`
- `{"question_edits": {"Q4a": {"text": "..."}, "Q3": {"criteria": {"Q3.c1": {"marks": 1.5}}}}}`

After an edit, the graph **re-validates** and **asks her again**, so she always approves exactly what gets stored.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">There are three <code>✋ TODO</code>s. Each is followed by a <b>check cell</b> that prints ✅ or ❌, so you know when you're done. The notebook runs top to bottom even before you start (you'll just see ❌s). Try it yourself first; the solution is in <code>week3/solutions/week3_EXERCISE_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

## Setup

In [ ]:
import json
import operator
from typing import Annotated, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

from gradewise.config import DATA_DIR, OUTPUT_DIR
from gradewise.exam.summary import exam_summary, print_exam_summary
from gradewise.schemas import ExamPaper

EXAM = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
PAPER = EXAM.model_dump()  # 💡 a plain dict, as it is in the graph's state


def check(label: str, ok: bool) -> None:
    print(f"{'✅' if ok else '❌'} {label}")


def question(paper: dict, qid: str) -> dict:
    return next(q for q in paper["questions"] if q["id"] == qid)


print(f"Loaded {EXAM.exam_id}: {len(EXAM.questions)} questions, problems: {EXAM.problems() or 'none'}")

## Part 1: Applying edits to one question

First, a plain function, no graph yet. It must **never mutate** the paper it's given (state is shared!), and the
result must still be a valid `ExamPaper`.

In [ ]:
def apply_question_edits(paper: dict, question_edits: dict) -> tuple[dict, list[str]]:
    """Apply Emma's edits to a COPY of `paper`. Returns (new_paper, errors).

    question_edits looks like {"Q4a": {"text": "..."}, "Q3": {"criteria": {"Q3.c1": {"marks": 1.5}}}}
    - top-level fields of a question (text, max_marks, methodology, ...) are replaced
    - "criteria" is a dict keyed by criterion ID: each one patches the matching criterion
    - an unknown question ID or criterion ID is an error (and changes nothing)
    - the edited paper must still pass ExamPaper.model_validate(); if not, return the ORIGINAL paper + errors
    """
    # ✋ TODO 1: implement this. Hints:
    #   • start with   new = copy.deepcopy(paper)   so the original is never mutated (day 3!)
    #   • find a question with   next((q for q in new["questions"] if q["id"] == qid), None)
    #   • for "criteria", loop over the patch dict and .update() the matching criterion dict
    #   • finish with   ExamPaper.model_validate(new)   inside try/except ValidationError
    return paper, []

### ✅ Check your work

In [ ]:
fixed_text = "What is a race condition? Give an example."
new, errors = apply_question_edits(PAPER, {"Q4a": {"text": fixed_text}})
check("edits a question's text", question(new, "Q4a")["text"] == fixed_text and not errors)
check("leaves the original paper untouched", question(PAPER, "Q4a")["text"] != fixed_text)

new, errors = apply_question_edits(PAPER, {"Q3": {"criteria": {"Q3.c1": {"marks": 1.5}}}})
check("patches one criterion by ID", question(new, "Q3")["criteria"][0]["marks"] == 1.5)
check(
    "...and the paper now has a problem to show Emma",
    "Q3: criteria add up to 3.5" in " ".join(ExamPaper.model_validate(new).problems()),
)

new, errors = apply_question_edits(PAPER, {"Q99": {"text": "?"}})
check("reports an unknown question", bool(errors) and "Q99" in errors[0])

new, errors = apply_question_edits(PAPER, {"Q3": {"criteria": {"Q3.c9": {"marks": 1}}}})
check("reports an unknown criterion", bool(errors) and "Q3.c9" in errors[0])

new, errors = apply_question_edits(PAPER, {"Q5": {"max_marks": -1}})
check(
    "rejects an edit that breaks the schema, keeping the original",
    bool(errors) and question(new, "Q5")["max_marks"] == 5,
)

## Part 2: The review graph

The state, plus the two nodes that don't change: `validate` (day 3's check, using `ExamPaper.problems()`) and
`store`, which writes the approved exam to `output/week3_exercise_exam.json`.

In [ ]:
class ReviewState(TypedDict, total=False):
    paper: dict
    problems: list[str]
    approved: bool | None
    edited: bool
    log: Annotated[list[str], operator.add]


def validate(state: ReviewState) -> dict:
    problems = ExamPaper.model_validate(state["paper"]).problems()
    return {"problems": problems, "log": [f"validate → {problems or 'clean ✅'}"]}


def store(state: ReviewState) -> dict:
    out = OUTPUT_DIR / "week3_exercise_exam.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(state["paper"], indent=2, ensure_ascii=False), encoding="utf-8")
    return {"log": [f"store → {out.name}"]}

Now the heart of it: `admin_review` must handle **edits** as well as approve/reject, and `after_review` must send
an edited exam back to `validate`.

In [ ]:
def admin_review(state: ReviewState) -> dict:
    # ⚠️ No side effects before interrupt(): this node re-runs from the top on resume.
    paper = ExamPaper.model_validate(state["paper"])
    decision = interrupt({"summary": exam_summary(paper), "problems": state["problems"]})

    # ✋ TODO 2: if the decision has "question_edits":
    #     • apply them with apply_question_edits(...)
    #     • return the new paper, edited=True, approved=None and a log line
    #     • if there were errors: keep the old paper, set edited=False and approved=None, and put the
    #       errors in "problems" so Emma can see what went wrong (after_review sends her straight back)
    #   otherwise: return approved=bool(decision.get("approved")) and edited=False
    return {"approved": bool(decision.get("approved")), "edited": False, "log": ["review → decision received"]}


def after_review(state: ReviewState) -> str:
    # ✋ TODO 3: edited → "validate" (check the edits, then ask Emma again)
    #           approved is None → "admin_review" (a bad edit: ask her again) · approved → "store" · otherwise END
    return "store" if state.get("approved") else END

In [ ]:
from gradewise.display import show_graph

def build_review_graph():
    builder = StateGraph(ReviewState)
    for node in (validate, admin_review, store):
        builder.add_node(node.__name__, node)
    builder.add_edge(START, "validate")
    builder.add_edge("validate", "admin_review")
    builder.add_conditional_edges("admin_review", after_review, ["validate", "admin_review", "store", END])
    builder.add_edge("store", END)
    return builder.compile(checkpointer=InMemorySaver())  # ⚠️ interrupt() needs a checkpointer


review = build_review_graph()
show_graph(review)

## Part 3: Emma's review session

Round 1: the graph pauses with the clean exam. Emma makes two edits: she fixes Q4a's wording and lowers Q3's
definition criterion to 1.5 marks, but **forgets** to give the extra half mark to another criterion.

In [ ]:
config = {"configurable": {"thread_id": "emma-review"}}
result = review.invoke({"paper": PAPER, "log": []}, config)
print("⏸ Round 1. Problems shown to Emma:", result["__interrupt__"][0].value["problems"] or "none")

result = review.invoke(
    Command(resume={"question_edits": {"Q4a": {"text": fixed_text}, "Q3": {"criteria": {"Q3.c1": {"marks": 1.5}}}}}),
    config,
)
round_2 = result.get("__interrupt__")
print(
    "⏸ Round 2. Problems shown to Emma:", round_2[0].value["problems"] if round_2 else "(the graph didn't pause again)"
)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Before running on: where should the half mark go? Q3.c2 and Q3.c3 are one mark each for naming a deadlock condition. What edit makes the criteria add up to 4 again? And what do you predict the graph will do after she sends it: store, or pause again?</span>
        </td>
    </tr>
</table>

In [ ]:
if round_2:
    result = review.invoke(Command(resume={"question_edits": {"Q3": {"criteria": {"Q3.c2": {"marks": 1.5}}}}}), config)
round_3 = result.get("__interrupt__")
print("⏸ Round 3. Problems shown to Emma:", (round_3[0].value["problems"] or "none ✅") if round_3 else "(no pause)")
if round_3:
    print_exam_summary(ExamPaper.model_validate(review.get_state(config).values["paper"]))

In [ ]:
if round_3:
    result = review.invoke(Command(resume={"approved": True}), config)
print("\n".join(result.get("log", [])))

stored = OUTPUT_DIR / "week3_exercise_exam.json"
final = ExamPaper.model_validate_json(stored.read_text(encoding="utf-8")) if stored.exists() and round_3 else None
check("the graph paused three times (edit → fix → approve)", bool(round_2 and round_3))
check(
    "the stored exam has Emma's Q4a wording", bool(final) and question(final.model_dump(), "Q4a")["text"] == fixed_text
)
check("the stored exam is clean", bool(final) and final.problems() == [])

### One more path: a bad edit
What if Emma types a negative mark? The edit must be refused, and she must see why.

In [ ]:
config = {"configurable": {"thread_id": "emma-typo"}}
review.invoke({"paper": PAPER, "log": []}, config)
result = review.invoke(Command(resume={"question_edits": {"Q5": {"max_marks": -5}}}), config)

pending = result.get("__interrupt__")
shown = pending[0].value["problems"] if pending else []
print("Shown to Emma:", shown)
check("a bad edit pauses again with the error", bool(pending) and any("max_marks" in p for p in shown))
check("the paper was not changed", question(review.get_state(config).values["paper"], "Q5")["max_marks"] == 5)

## Recap

You gave the human in the loop real power: not just yes or no, but **targeted edits**, applied to a copy,
**re-validated** against the schema and the marking rules, and shown back to her before anything is stored.
That's the difference between a rubber stamp and a genuine review.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">"Reject and re-run" costs a model call and the reviewer's patience; "edit in place" costs neither. Small design choices like this decide whether busy teachers, or claims handlers, or auditors, actually <i>use</i> the AI system you build, or quietly go back to the spreadsheet.</span>
        </td>
    </tr>
</table>

⏭ **Next:** week 4, `week4/day1.ipynb`: agentic RAG, and reading Sam's handwriting.